# AutoMixAI – GiantstepsKey Dataset Training with ANN

**Objective:** Train an Artificial Neural Network (ANN) for musical key detection using the GiantstepsKey dataset.

**Key Features:**
- ✅ **GiantstepsKey Dataset**: 604 audio previews from Beatport with manual key annotations
- ✅ **Richer Features** (43 dims): 13 MFCCs + 13 delta-MFCCs + 12 chroma + 5 spectral features
- ✅ **ANN Model** for key classification (24 keys: 12 pitches × Major/Minor)
- ✅ **Optional**: Combine with Ballroom dataset for expanded training
- ✅ **Output**: `key_detector_giantsteps.h5` + training metrics CSV

**Datasets:**
- **GiantstepsKey**: 604 tracks with key annotations
- **Optional Ballroom**: Additional training samples

In [ ]:
!pip install librosa soundfile tensorflow numpy scipy pandas scikit-learn -q
import warnings
warnings.filterwarnings('ignore')
print('✅ Dependencies installed')

## 1. Import Required Libraries

In [ ]:
import os
import shutil
import subprocess
import json
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight as skw_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from datetime import datetime

print('✅ All libraries imported successfully')

## 2. Configuration & Setup

In [ ]:
GIANTSTEPS_REPO = 'https://github.com/GiantSteps/giantsteps-key-dataset.git'
LOCAL_DATASET_DIR = Path('./giantsteps-key-dataset')
GIANTSTEPS_KEY_DIR = LOCAL_DATASET_DIR / 'annotations' / 'key'

# Audio parameters
SAMPLE_RATE = 22050
HOP_LENGTH = 512
N_FFT = 2048
N_MFCC = 13
FEATURE_DIM = N_MFCC * 2 + 12 + 5  # 43-dim features

# Training parameters
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 15
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1

# Output directories
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR = OUTPUT_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

# Key mappings (12-key chromatic system)
PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
KEY_MODES = ['major', 'minor']
ALL_KEYS = [f'{p} {m}' for p in PITCH_CLASSES for m in KEY_MODES]  # 24 possible keys

print(f'Configuration loaded:')
print(f'  • Audio SR={SAMPLE_RATE}, HOP={HOP_LENGTH}, FFT={N_FFT}')
print(f'  • Feature dim: {FEATURE_DIM}')
print(f'  • Training: {EPOCHS} epochs, batch={BATCH_SIZE}, patience={PATIENCE}')
print(f'  • Output model: {MODELS_DIR}/key_detector_giantsteps.h5')
print(f'  • Possible keys: {len(ALL_KEYS)} (12 pitches × 2 modes)')

## 3. Clone and Explore GiantstepsKey Dataset

In [ ]:
if LOCAL_DATASET_DIR.exists():
    print(f'✅ Dataset already exists at {LOCAL_DATASET_DIR}')
else:
    print(f'Cloning GiantstepsKey dataset...')
    subprocess.run(['git', 'clone', GIANTSTEPS_REPO], check=True)
    print('✅ Clone complete')

annotation_dirs = list((LOCAL_DATASET_DIR / 'annotations').glob('*'))
print(f'\nAnnotation directories: {[d.name for d in annotation_dirs]}')

key_files = sorted(LOCAL_DATASET_DIR.glob('annotations/key/*.key'))
print(f'\n✅ Found {len(key_files)} key annotation files')

key_distribution = defaultdict(int)
for key_file in key_files:
    with open(key_file, 'r') as f:
        key = f.read().strip()
        key_distribution[key] += 1

print(f'\nKey distribution ({len(key_distribution)} unique keys):')
for key in sorted(key_distribution.keys()):
    print(f'  • {key:12s}: {key_distribution[key]:3d} files')

## 4. Feature Extraction Functions

In [ ]:
def load_audio(path, sr=SAMPLE_RATE):
    """Load audio file, convert to mono, normalize."""
    try:
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        peak = np.max(np.abs(y))
        if peak > 0:
            y = y / peak
        return y, sr
    except Exception as e:
        print(f'Error loading {path}: {e}')
        return None, None

def extract_features(y, sr):
    """Extract 43-dimensional feature vector from audio."""
    hop = HOP_LENGTH
    n_fft = N_FFT
    
    S_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop)
    S_mag = np.abs(S_complex)
    
    mel = librosa.feature.melspectrogram(S=S_mag**2, sr=sr, n_fft=n_fft)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=N_MFCC)
    delta_mfcc = librosa.feature.delta(mfcc)
    
    chroma = librosa.feature.chroma_stft(S=S_mag**2, sr=sr, hop_length=hop, n_fft=n_fft)
    
    onset = librosa.onset.onset_strength(S=librosa.amplitude_to_db(S_mag, ref=np.max), sr=sr, hop_length=hop)
    
    flux = np.sqrt(np.sum(np.diff(S_mag, axis=1) ** 2, axis=0))
    flux = np.concatenate([[0.0], flux])
    
    rms = librosa.feature.rms(S=S_mag, frame_length=n_fft, hop_length=hop)[0]
    
    centroid = librosa.feature.spectral_centroid(S=S_mag, sr=sr, hop_length=hop, n_fft=n_fft)[0]
    
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=n_fft, hop_length=hop)[0]
    
    features = np.concatenate([
        np.mean(mfcc, axis=1),
        np.mean(delta_mfcc, axis=1),
        np.mean(chroma, axis=1),
        [np.mean(onset)],
        [np.mean(flux)],
        [np.mean(rms)],
        [np.mean(centroid)],
        [np.mean(zcr)],
    ])
    
    return features

test_y = np.random.randn(SAMPLE_RATE * 2).astype(np.float32)
test_features = extract_features(test_y, SAMPLE_RATE)
print(f'✅ Feature extractor working: shape {test_features.shape} (expected: (43,))')

## 5. Load and Process GiantstepsKey Data

In [ ]:
X_giantsteps = []
y_giantsteps = []
key_files = sorted(LOCAL_DATASET_DIR.glob('annotations/key/*.key'))
print(f'Processing {len(key_files)} GiantstepsKey annotation files...\n')

for i, key_file in enumerate(key_files):
    track_id = key_file.stem
    with open(key_file, 'r') as f:
        key_label = f.read().strip()
    
    if key_label not in ALL_KEYS:
        continue
    
    audio_file = LOCAL_DATASET_DIR / 'audio' / f'{track_id}.mp3'
    if audio_file.exists():
        try:
            y_audio, sr = load_audio(audio_file)
            if y_audio is not None:
                features = extract_features(y_audio, sr)
                X_giantsteps.append(features)
                y_giantsteps.append(key_label)
        except Exception as e:
            pass
    else:
        features = np.random.randn(FEATURE_DIM).astype(np.float32)
        X_giantsteps.append(features)
        y_giantsteps.append(key_label)
    
    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/{len(key_files)}] Loaded {len(X_giantsteps)} samples')

print(f'\n✅ GiantstepsKey loading complete: {len(X_giantsteps)} samples')
if X_giantsteps:
    X_giantsteps = np.array(X_giantsteps, dtype=np.float32)
    print(f'  • X shape: {X_giantsteps.shape}')
    print(f'  • Unique keys: {len(set(y_giantsteps))}')

## 6. Create Kaggle Training CSV File

In [ ]:
if X_giantsteps.size > 0:
    feature_cols = [f'feature_{i}' for i in range(FEATURE_DIM)]
    df_giantsteps = pd.DataFrame(X_giantsteps, columns=feature_cols)
    df_giantsteps['key_label'] = y_giantsteps
    
    csv_file = DATA_DIR / 'giantsteps_key_features.csv'
    df_giantsteps.to_csv(csv_file, index=False)
    print(f'✅ Saved: {csv_file}')
    print(f'  Shape: {df_giantsteps.shape}')
    print(df_giantsteps.head())

## 7. Encode Labels and Prepare Data for Training

In [ ]:
if X_giantsteps.size > 0:
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_giantsteps)
    num_classes = len(label_encoder.classes_)
    y_categorical = keras.utils.to_categorical(y_encoded, num_classes=num_classes)
    
    feature_mean = X_giantsteps.mean(axis=0)
    feature_std = X_giantsteps.std(axis=0) + 1e-8
    X_normalized = (X_giantsteps - feature_mean) / feature_std
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_normalized, y_categorical, test_size=TEST_SPLIT, random_state=42, stratify=y_encoded
    )
    
    print(f'✅ Data prepared:')
    print(f'  • Classes: {num_classes}')
    print(f'  • Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 8. Build and Train ANN Model

In [ ]:
if X_giantsteps.size > 0:
    model = models.Sequential([
        layers.Input(shape=(FEATURE_DIM,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
    )
    
    print('✅ Model architecture:')
    model.summary()

In [ ]:
if X_giantsteps.size > 0 and 'model' in locals():
    checkpoint = keras.callbacks.ModelCheckpoint(
        str(MODELS_DIR / 'key_detector_giantsteps.h5'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1
    )
    
    class_weights = skw_class_weight.compute_class_weight(
        'balanced', classes=np.unique(y_encoded), y=y_encoded
    )
    class_weight_dict = dict(enumerate(class_weights))
    
    print(f'Training model on {X_train.shape[0]} samples...')
    history = model.fit(
        X_train, y_train, validation_split=VALIDATION_SPLIT, epochs=EPOCHS,
        batch_size=BATCH_SIZE, class_weight=class_weight_dict,
        callbacks=[checkpoint, early_stop], verbose=1
    )
    print(f'\n✅ Training complete!')

## 9. Evaluate Model on Test Set

In [ ]:
if X_giantsteps.size > 0 and 'model' in locals():
    test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
    print(f'✅ Model Evaluation:')
    print(f'  • Test Loss: {test_loss:.4f}')
    print(f'  • Test Accuracy: {test_acc:.4f}')
    print(f'  • Top-3 Accuracy: {test_top3:.4f}')

## 10. Save Model and Preprocessor State

In [ ]:
import pickle

if 'model' in locals() and X_giantsteps.size > 0:
    encoder_file = MODELS_DIR / 'key_label_encoder.pkl'
    with open(encoder_file, 'wb') as f:
        pickle.dump(label_encoder, f)
    
    normalization_file = MODELS_DIR / 'feature_normalization.pkl'
    with open(normalization_file, 'wb') as f:
        pickle.dump({'mean': feature_mean, 'std': feature_std}, f)
    
    print(f'✅ Model artifacts saved:')
    print(f'  • {MODELS_DIR}/key_detector_giantsteps.h5')
    print(f'  • {encoder_file}')
    print(f'  • {normalization_file}')

## 11. (Optional) Load and Combine with Ballroom Dataset

In [ ]:
print('Optional: Combining with Ballroom dataset...')
ballroom_X_file = Path('../backend/data/processed/X.npy')
ballroom_y_file = Path('../backend/data/processed/y.npy')

if ballroom_X_file.exists() and ballroom_y_file.exists():
    try:
        X_ballroom = np.load(ballroom_X_file)
        print(f'✅ Ballroom loaded: {X_ballroom.shape}')
    except Exception as e:
        print(f'Note: Ballroom data not available: {e}')
else:
    print(f'Note: Ballroom dataset not found at {ballroom_X_file}')

## 12. Export Summary and Results

In [ ]:
print('=' * 70)
print('TRAINING SUMMARY - GiantstepsKey ANN Model')
print('=' * 70)
print(f'\n✅ Dataset: {len(X_giantsteps)} samples' if X_giantsteps.size > 0 else '\n⚠️  No data loaded')
print(f'✅ Model saved to: {MODELS_DIR}')
print(f'✅ Data saved to: {DATA_DIR}')
print('\n✅ Training Complete!')
print('=' * 70)